## Student Name: Gabriel Espinoza Rojas

## Student ID: 22176691

# Task 1

## Task 1.A

Use the following code snippet to download <a href="https://norvig.com/ngrams/count_1w.txt">Peter Norvig's unigram corpus</a>, and load it into a dataframe, <code>unigrams_df</code>.

In [ ]:
import pandas as pd

#install and import the jellyfish library
!pip install jellyfish
import jellyfish

!wget https://norvig.com/ngrams/count_1w.txt   #unigram corpus
filePath1 = "/content/count_1w.txt"
unigrams_df = pd.read_csv(filePath1, sep='\t', header=None, names=['unigram','count'], keep_default_na=False)  #keep_default_na to be set here to avoid NaNs

print(f'Number of unigrams: {unigrams_df.shape[0]}')
display(unigrams_df.head(100))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 1.4 MB/s eta 0:00:00a 0:00:01m
--2026-10-10 01:23:18--  https://norvig.com/ngrams/count_1w.txt
Resolving norvig.com (norvig.com)... 158.106.138.13
Connecting to norvig.com (norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.norvig.com/ngrams/count_1w.txt [following]
--2026-10-10 01:23:20--  https://www.norvig.com/ngrams/count_1w.txt
Resolving www.norvig.com (www.norvig.com)... 158.106.138.13
Connecting to www.norvig.com (www.norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4956241 (4.7M) [text/plain]
Saving to: ‘count_1w.txt’

count_1w.txt        100%[===================>]   4.73M  2.72MB/s    in 1.7s    

2026-10-10 01:23:23 (2.72 MB/s) - ‘count_1w.txt’ saved [4956241/4956241]

Number of unigrams: 333333


,unigram,count
0,the,23135851162
1,of,13151942776
2,and,12997637966
3,to,12136980858
4,a,9081174698
...,...,...
95,like,520585287
96,service,519537222
97,x,508609523
98,than,502609275


## Task 1.B

Write a simple spelling correction function that takes a <b>non-word string</b> as input (i.e. non-word misspellings such as "acress"), and prints out a list of candidate corrections ranked by their prior, <b>P(w)</b>.

<b>Hint 1:</b> only use the language model/priors, i.e. the unigram counts stored in <code>unigrams_df</code>, and <u>ignore</u> the channel/error model probabilities.

<b>Hint 2:</b> Use the <b>Damerau-Levenshtein Distance</b> function in the <a href="https://github.com/jamesturk/jellyfish">jellyfish library</a> for candidate generation.

<b>Expected Output:</b>

```
nonWord= acress

      word    frequency    P(word)
 0    ...     ...          ...
 1    ...     ...          ...
 ...
```

In [5]:
def nonWordSpellingCorrection(nonWord):
    """
    Return a dataframe of candidate corrections for `nonWord`, ranked by prior P(w)
    """

    print(f"nonWord= {nonWord}\n")

    # to calculate unigram probability
    N = unigrams_df['count'].sum()

    # lambda to calculate the Damerau Levenshtein distances of all words
    distances = unigrams_df['unigram'].apply(
        lambda w: jellyfish.damerau_levenshtein_distance(nonWord, w)
    )

    # leave only the candidates (minimum distance)
    candidates_df = unigrams_df.loc[distances == 1].copy()

    # unigram probability: P(w) = count(w) / N
    candidates_df['P(word)'] = candidates_df['count'] / N

    # ranking by their prior, P(w) and also resetting index
    candidates_df = candidates_df.sort_values('P(word)', ascending=False).reset_index(drop=True)

    # renaming columns to match expected output
    candidates_df = candidates_df.rename(columns={'unigram': 'word', 'count': 'frequency'})
    
    # prints out the list of candidate corrections 
    print(candidates_df)

    return candidates_df

candidates_df = nonWordSpellingCorrection("acress")

nonWord= acress

      word  frequency       P(word)
0   access  217986984  3.706479e-04
1   across   76597151  1.302397e-04
2    acres   14208905  2.415970e-05
3  actress    7010056  1.191935e-05
4   adress     984657  1.674233e-06
5   caress     590047  1.003269e-06
6    cress     279364  4.750085e-07
7   apress     202431  3.441977e-07
8    acess     171785  2.920897e-07


Instead of relying on explicit loops, I tried to use pandas as much as possible to maximise the advantage of using the library. 

Notice that I'm using ED = 1, the instructions ask for a <strong>simple</strong> spelling function. Dan Jurafsky mentions in [this video](https://www.youtube.com/watch?v=jk8jjRZUpf8&t=480s) that 80% of errors are at this distance. Is it fair to then use that as a rule of thumb? He also mentions that all errors are within an edit distance of 2, but his examples only considered distances of 1. Is 80% enough? I ran the same code but allowing also words with a Damerau-Levenshtein distance of 2 to find out:

In [6]:
# just a little experiment
def nonWordSpellingCorrectionED2(nonWord):
    """
    Return a dataframe of candidate corrections for `nonWord`, ranked by prior P(w)
    """

    print(f"nonWord= {nonWord}\n")

    # to calculate unigram probability
    N = unigrams_df['count'].sum()

    # lambda to calculate the Damerau Levenshtein distances of all words
    distances = unigrams_df['unigram'].apply(
        lambda w: jellyfish.damerau_levenshtein_distance(nonWord, w)
    )

    # leave only the candidates (minimum distance)
    candidates_df = unigrams_df.loc[(distances == 1) | (distances == 2)].copy()

    # # unigram probability: P(w) = count(w) / N
    candidates_df['P(word)'] = candidates_df['count'] / N

    # ranking by their prior, P(w) and also resetting index
    candidates_df = candidates_df.sort_values('P(word)', ascending=False).reset_index(drop=True)

    # renaming columns to match expected output
    candidates_df = candidates_df.rename(columns={'unigram': 'word', 'count': 'frequency'})
    
    # prints out the list of candidate corrections 
    print(candidates_df)

    return candidates_df

candidates_df = nonWordSpellingCorrectionED2("acress")

nonWord= acress

        word  frequency       P(word)
0    address  261872866  4.452680e-04
1     access  217986984  3.706479e-04
2      press  179652730  3.054673e-04
3      areas  121986327  2.074159e-04
4     across   76597151  1.302397e-04
..       ...        ...           ...
136   peress      14267  2.425848e-08
137    fress      14017  2.383340e-08
138   adrese      13298  2.261087e-08
139    cresp      12963  2.204126e-08
140   epress      12722  2.163148e-08

[141 rows x 3 columns]


I think the rule of thumb Dan uses is fair and I'm sticking to my original function. The output considering ED = 2 as well as ED = 1 returns an overwhelming amount of rows. And despite finding words with high probability that did not appear at ED = 1 (address, press, areas...), because we are using a unigram (and no channel model), this only means these words have high frequency in our corpus, it does not automatically mean they are a better fit. 

# Task 2

## Task 2.A

Use the <a href="https://textblob.readthedocs.io/en/dev/">TextBlob Library</a> to write a function that takes a string (e.g. <i>acress</i>) as input and returns a list of <b>(word, confidence)</b> tuples with spelling suggestions.

<b>Sample output:</b>

```
string= acress
candidates= [('across', 0.6851851851851852), ('access', 0.1728395061728395), ('acres', 0.1111111111111111), ('actress', 0.021604938271604937), ('caress', 0.009259259259259259), ...]
```

In [ ]:
!pip install textblob
from textblob import Word


def textBlobSuggestions(string):
    """
    Returns the list of (word, confidence) tuples TextBlob proposes for `string`
    """
    
    print(f"string= {string}")

    # https://textblob.readthedocs.io/en/dev/quickstart.html#spelling-correction
    w = Word(string)
    return w.spellcheck()


candidates = textBlobSuggestions("acress")
print(f"candidates= {candidates}")

string= acress
candidates= [('across', 0.6851851851851852), ('access', 0.1728395061728395), ('acres', 0.1111111111111111), ('actress', 0.021604938271604937), ('caress', 0.009259259259259259)]


The output of this library also supports my decision of the ED in task 1.B, where the output is a short list that superficially I can see looks like it is using ED = 1 under the hood.

I checked the source code and it is a little bit more complex than that, TextBlob prefers ED=1 and only widens to ED=2 when ED=1 yields no known words:
```
candidates = (
    self._known([w])
    or self._known(self._edit1(w))
    or self._known(self._edit2(w))
    or [w]
)
```
https://github.com/sloria/TextBlob/blob/e8401fa714b33157c4fa8e2e9233d9fec6cb00ef/src/textblob/_text.py#L1688

## Task 2.B

Use the <a href="https://textblob.readthedocs.io/en/dev/">TextBlob Library</a> to write a function that takes a <b>sentence</b> as input and corrects its spelling.

<b>Sample output:</b>

```
sentence= I have good speling
Corrected sentence= I have good spelling
```

In [43]:
from textblob import TextBlob


def textBlobCorrectSentence(sentence):
    """
    Prints the original input sentence and the TextBlob's sentence with the spelling corrected.
    """
    print(f"sentence= {sentence}")

    # https://textblob.readthedocs.io/en/dev/quickstart.html#spelling-correction
    b = TextBlob(sentence)
    correct_sentence = b.correct()
    print(f"Corrected sentence= {correct_sentence}")

textBlobCorrectSentence("I have good speling")

sentence= I have good speling
Corrected sentence= I have good spelling
